In [1]:
# ─────────────────────────────────────────────
# Imports & dependency installation (run this cell first)
# ─────────────────────────────────────────────

import subprocess
packages = [
    "bitsandbytes>=0.46.1",
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "trl>=0.8.0",
    "accelerate>=0.27.0",
    "datasets>=2.18.0",
]
for pkg in packages:
    subprocess.run(["pip", "install", "-q", pkg], check=True)

import csv
import json
import os
import re
from dataclasses import dataclass, field
from typing import Optional

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
from trl import SFTTrainer, SFTConfig

print("Imports and dependencies ready.")

In [ ]:
# ─────────────────────────────────────────────
# Data Structures
# ─────────────────────────────────────────────

@dataclass
class Question:
    """Represents a single question/event sample from questions.jsonl"""
    id: str  # e.g. "q-1" — required for Codabench submission
    topic_id: int  # links to docs.json; int in dataset
    target_event: str
    option_a: str
    option_b: str
    option_c: str
    option_d: str
    golden_answer: Optional[list[str]] = None  # None for test split (no labels)

    @property
    def options(self) -> dict:
        return {
            "A": self.option_a,
            "B": self.option_b,
            "C": self.option_c,
            "D": self.option_d,
        }


@dataclass
class Document:
    """Represents a single contextual document from docs.json"""
    doc_id: str
    title: str
    text: str
    source: Optional[str] = None


@dataclass
class Sample:
    """A fully linked sample: question + its context documents"""
    question: Question
    documents: list[Document] = field(default_factory=list)


# ─────────────────────────────────────────────
# Loader Functions
# ─────────────────────────────────────────────

def load_questions(filepath: str) -> list[Question]:
    """Load questions from a .jsonl file (one JSON object per line)."""
    questions = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)

            # Golden answer may be missing in test data
            golden = obj.get("golden_answer", None)
            # Normalize to list if it's a string like "A" or "A,B"
            if isinstance(golden, str):
                golden = [g.strip() for g in golden.split(",")]

            q = Question(
                id=obj["id"],
                topic_id=obj["topic_id"],
                target_event=obj["target_event"],
                option_a=obj["option_A"],
                option_b=obj["option_B"],
                option_c=obj["option_C"],
                option_d=obj["option_D"],
                golden_answer=golden,
            )
            questions.append(q)
    return questions


def load_docs(filepath: str) -> dict[int, list[Document]]:
    """
    Load documents from docs.json.
    docs.json is a list of records: [ {"topic_id": 1, "topic": "...", "docs": [...]}, ... ]
    Returns a dict mapping topic_id -> list of Document objects.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        raw = json.load(f)

    docs_by_topic: dict[int, list[Document]] = {}
    for record in raw:
        topic_id = record["topic_id"]
        doc_list = record["docs"]
        docs = []
        for d in doc_list:
            doc = Document(
                doc_id=d.get("doc_id", d.get("id", "")),
                title=d.get("title", ""),
                text=d.get("text", d.get("content", "")),
                source=d.get("source", None),
            )
            docs.append(doc)
        docs_by_topic[topic_id] = docs
    return docs_by_topic


def load_split(split_dir: str) -> list[Sample]:
    """
    Load a full data split (train/dev/test/sample) from its directory.
    Expects: questions.jsonl and docs.json inside split_dir.
    Returns a list of fully linked Sample objects.
    """
    questions_path = os.path.join(split_dir, "questions.jsonl")
    docs_path = os.path.join(split_dir, "docs.json")

    questions = load_questions(questions_path)
    docs_by_topic = load_docs(docs_path)

    samples = []
    for q in questions:
        docs = docs_by_topic.get(q.topic_id, [])
        samples.append(Sample(question=q, documents=docs))

    return samples


# ─────────────────────────────────────────────
# Main: Load All Splits
# ─────────────────────────────────────────────

def load_all_splits(base_dir: str) -> dict[str, list[Sample]]:
    """Load train, dev, test, and sample splits from the dataset root directory."""
    splits = {
        "train": "train_data",
        "dev":   "dev_data",
        "test":  "test_data",
        "sample": "sample_data",
    }

    all_data = {}
    for split_name, folder in splits.items():
        split_path = os.path.join(base_dir, folder)
        if os.path.isdir(split_path):
            all_data[split_name] = load_split(split_path)
            print(f"[✓] Loaded '{split_name}': {len(all_data[split_name])} samples")
        else:
            print(f"[!] Skipped '{split_name}': folder not found at {split_path}")

    return all_data


# ─────────────────────────────────────────────
# Quick Inspection Helper
# ─────────────────────────────────────────────

def inspect_sample(sample: Sample):
    """Print a human-readable summary of a single sample."""
    q = sample.question
    print("=" * 60)
    print(f"ID           : {q.id}")
    print(f"Topic ID     : {q.topic_id}")
    print(f"Target Event : {q.target_event}")
    print(f"Options:")
    for key, val in q.options.items():
        print(f"  [{key}] {val}")
    print(f"Golden Answer: {q.golden_answer}")
    print(f"Docs linked  : {len(sample.documents)}")
    if sample.documents:
        first_doc = sample.documents[0]
        print(f"  First doc title : {first_doc.title}")
        print(f"  First doc text  : {first_doc.text[:200]}...")
    print("=" * 60)


# ─────────────────────────────────────────────
# Entry Point
# ─────────────────────────────────────────────

if __name__ == "__main__":
    # ← Update this path to your local dataset root
    # BASE_DIR = "semeval2026-task12-dataset-main"
    BASE_DIR = "/kaggle/input/datasets/nosadeghob/semeval2026-task12-dataset-main/semeval2026-task12-dataset-main"

    print("Loading dataset...\n")
    data = load_all_splits(BASE_DIR)

    # Inspect the first training sample as a sanity check
    if "train" in data and data["train"]:
        print("\n--- Sample Inspection (first train item) ---")
        inspect_sample(data["train"][0])

    # Summary statistics
    print("\n--- Dataset Summary ---")
    for split, samples in data.items():
        has_labels = sum(1 for s in samples if s.question.golden_answer is not None)
        print(f"{split:8s}: {len(samples):4d} samples | {has_labels:4d} labeled")

In [ ]:
# ─────────────────────────────────────────────
# Evaluation Metric
# ─────────────────────────────────────────────

def score_sample(predicted: list[str], golden: list[str]) -> float:
    """
    Score a single prediction against the golden answer.

    Args:
        predicted : list of predicted option labels, e.g. ["A"] or ["A", "B"]
        golden    : list of correct option labels,   e.g. ["D"] or ["A", "C"]

    Returns:
        1.0  — Full Match   : predicted == golden (same set)
        0.5  — Partial Match: predicted is a non-empty subset of golden (no wrong options)
        0.0  — Incorrect    : predicted is empty, contains wrong options, or doesn't match
    """
    P = set(predicted)
    G = set(golden)

    if not P:                    # empty prediction
        return 0.0
    if P == G:                   # exact match
        return 1.0
    if P.issubset(G):            # subset with no wrong options
        return 0.5
    return 0.0                   # wrong option included or no overlap


def evaluate(predictions: dict[str, list[str]], samples: list) -> dict:
    """
    Evaluate model predictions against a labeled split.

    Args:
        predictions : dict mapping question id -> list of predicted labels
                      e.g. {"q-201": ["D"], "q-202": ["A", "C"], ...}
        samples     : list of Sample objects (must have golden_answer)

    Returns:
        dict with 'score', 'full_matches', 'partial_matches', 'incorrect', 'total'
    """
    total         = 0
    full_matches  = 0
    partial_matches = 0
    incorrect     = 0
    score_sum     = 0.0

    for sample in samples:
        q = sample.question
        if q.golden_answer is None:
            continue  # skip unlabeled test samples

        predicted = predictions.get(q.id, [])
        s = score_sample(predicted, q.golden_answer)

        score_sum += s
        total += 1

        if s == 1.0:
            full_matches += 1
        elif s == 0.5:
            partial_matches += 1
        else:
            incorrect += 1

    final_score = score_sum / total if total > 0 else 0.0

    results = {
        "score"           : round(final_score, 4),
        "full_matches"    : full_matches,
        "partial_matches" : partial_matches,
        "incorrect"       : incorrect,
        "total"           : total,
    }

    print(f"Final Score     : {results['score']}")
    print(f"Full Matches    : {full_matches}  ({100*full_matches/total:.1f}%)")
    print(f"Partial Matches : {partial_matches}  ({100*partial_matches/total:.1f}%)")
    print(f"Incorrect       : {incorrect}  ({100*incorrect/total:.1f}%)")
    print(f"Total Samples   : {total}")

    return results


# ─────────────────────────────────────────────
# Sanity Check
# ─────────────────────────────────────────────

# Simulate a few predictions to verify scoring works correctly
test_cases = [
    (["D"],      ["D"],      1.0,  "Full match"),
    (["A", "C"], ["A", "C"], 1.0,  "Full match - multi label"),
    (["A"],      ["A", "C"], 0.5,  "Partial match - subset"),
    (["A", "B"], ["A", "C"], 0.0,  "Incorrect - contains wrong option"),
    ([],         ["D"],      0.0,  "Incorrect - empty prediction"),
    (["B"],      ["D"],      0.0,  "Incorrect - wrong option"),
]

print("--- Scoring Sanity Checks ---")
all_passed = True
for pred, gold, expected, desc in test_cases:
    result = score_sample(pred, gold)
    status = "✓" if result == expected else "✗"
    if result != expected:
        all_passed = False
    print(f"  [{status}] {desc}: score={result} (expected {expected})")

print(f"\nAll checks passed: {all_passed}")

In [ ]:
# ─────────────────────────────────────────────
# Step 3: Prompt Formatting
# ─────────────────────────────────────────────

def format_docs(documents: list, max_docs: int = 5, max_chars_per_doc: int = 500) -> str:
    """
    Convert a list of Document objects into a single context string.
    We limit to top max_docs and truncate each doc to keep the prompt manageable.
    """
    if not documents:
        return "No context documents available."

    parts = []
    for i, doc in enumerate(documents[:max_docs]):
        text = doc.text[:max_chars_per_doc].strip()
        title = doc.title.strip() if doc.title else "Untitled"
        parts.append(f"[Document {i+1}] {title}\n{text}")

    return "\n\n".join(parts)


def format_prompt(sample: Sample, include_answer: bool = True) -> str:
    """
    Format a Sample into a full instruction prompt.

    If include_answer=True  → used for fine-tuning (includes the correct answer)
    If include_answer=False → used for inference (model must generate the answer)
    """
    q = sample.question
    context = format_docs(sample.documents)

    prompt = f"""You are an expert in causal reasoning. Given a real-world event and contextual documents, identify the most likely DIRECT cause(s) of the event from the options provided.

### Context Documents:
{context}

### Event:
{q.target_event}

### Options:
A) {q.option_a}
B) {q.option_b}
C) {q.option_c}
D) {q.option_d}

### Instructions:
- Select the option(s) that are the most likely DIRECT cause of the event.
- You may select one or more options if multiple causes apply.
- Reply with ONLY the letter(s) of your answer, separated by commas. Example: A  or  A,C

### Answer:"""

    if include_answer and q.golden_answer:
        answer = ",".join(sorted(q.golden_answer))
        prompt += f" {answer}"

    return prompt


def build_dataset_for_finetuning(samples: list[Sample]) -> list[dict]:
    """
    Convert a list of Sample objects into a list of dicts with 'text' field,
    ready to be consumed by HuggingFace datasets / SFTTrainer.
    """
    records = []
    for sample in samples:
        if sample.question.golden_answer is None:
            continue  # skip unlabeled samples
        text = format_prompt(sample, include_answer=True)
        records.append({"text": text})
    return records


# ─────────────────────────────────────────────
# Build formatted splits
# ─────────────────────────────────────────────

train_records = build_dataset_for_finetuning(data["train"])
dev_records   = build_dataset_for_finetuning(data["dev"])

print(f"Train records : {len(train_records)}")
print(f"Dev records   : {len(dev_records)}")

# ─────────────────────────────────────────────
# Sanity check: print one example
# ─────────────────────────────────────────────
print("\n--- Example Fine-tuning Prompt ---\n")
print(train_records[0]["text"])

In [ ]:
# ─────────────────────────────────────────────
# Step 4: Load Model with QLoRA
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # change if preferred
MAX_SEQ_LENGTH = 2048   # max token length per sample

# QLoRA: load model in 4-bit to fit in 16GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   # extra memory saving
    bnb_4bit_quant_type="nf4",        # normalized float 4 — best quality
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ─────────────────────────────────────────────
# Load Tokenizer
# ─────────────────────────────────────────────

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token       # required for batched training
tokenizer.padding_side = "right"                # pad on right for causal LM

# ─────────────────────────────────────────────
# Load Model in 4-bit
# ─────────────────────────────────────────────

print("Loading model in 4-bit (QLoRA)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",              # automatically use available GPU(s)
    trust_remote_code=True,
)

# Required step before attaching LoRA to a quantized model
model = prepare_model_for_kbit_training(model)

# ─────────────────────────────────────────────
# Attach LoRA Adapters
# ─────────────────────────────────────────────

lora_config = LoraConfig(
    r=16,                          # rank — higher = more capacity, more memory
    lora_alpha=32,                 # scaling factor (usually 2x r)
    target_modules=[               # which layers to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# ─────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────

model.print_trainable_parameters()

# Verify GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f"\nGPU Memory — Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")

In [ ]:
# ─────────────────────────────────────────────
# Step 5 (fixed v3): Fine-tuning with SFTTrainer
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
# Convert to HuggingFace Dataset
# ─────────────────────────────────────────────

# train_records_subset = train_records[:900]   # ~27% of data
# hf_train = Dataset.from_list(train_records_subset)
hf_train = Dataset.from_list(train_records)
hf_dev   = Dataset.from_list(dev_records)

print(f"HF Train size : {len(hf_train)}")
print(f"HF Dev size   : {len(hf_dev)}")

# ─────────────────────────────────────────────
# SFTConfig
# ─────────────────────────────────────────────

sft_config = SFTConfig(
    output_dir="./qlora-abductive",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    fp16=False,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    packing=False,
    max_length=MAX_SEQ_LENGTH,   # back in SFTConfig
)

# ─────────────────────────────────────────────
# SFTTrainer
# ─────────────────────────────────────────────

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=hf_train,
    eval_dataset=hf_dev,
    args=sft_config,
)

# ─────────────────────────────────────────────
# Train
# ─────────────────────────────────────────────

print("\nStarting fine-tuning...\n")
trainer.train()

# ─────────────────────────────────────────────
# Save the final LoRA adapter
# ─────────────────────────────────────────────

ADAPTER_DIR = "./qlora-abductive-final"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\nLoRA adapter saved to: {ADAPTER_DIR}")

In [ ]:
# ─────────────────────────────────────────────
# VRAM Usage Check
# ─────────────────────────────────────────────
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    peak      = torch.cuda.max_memory_allocated() / 1e9   # peak since last reset
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM       : {total:.2f} GB")
    print(f"Currently used   : {allocated:.2f} GB")
    print(f"Reserved (cache) : {reserved:.2f} GB")
    print(f"Peak allocated   : {peak:.2f} GB")
    print(f"Free             : {total - reserved:.2f} GB")

In [ ]:
# ─────────────────────────────────────────────
# Step 6: Inference on Test Set
# ─────────────────────────────────────────────


model.eval()

# ─────────────────────────────────────────────
# Answer extraction helper
# ─────────────────────────────────────────────

def extract_answer(generated_text: str) -> list[str]:
    """
    Extract answer letters from the model's generated text.
    Looks for patterns like: A  /  A,B  /  A, B  /  A and B
    Returns a sorted list of unique valid letters.
    """
    if "### Answer:" in generated_text:
        answer_part = generated_text.split("### Answer:")[-1]
    else:
        answer_part = generated_text

    letters = re.findall(r'\b([A-D])\b', answer_part)
    letters = sorted(set(letters))

    return letters if letters else ["A"]


# ─────────────────────────────────────────────
# Inference loop
# ─────────────────────────────────────────────

def run_inference(samples: list) -> dict[str, list[str]]:
    """
    Run inference on a list of Sample objects.
    Returns a dict mapping question id (unique per sample) -> predicted labels.
    """
    predictions = {}

    for sample in tqdm(samples, desc="Running inference"):
        q = sample.question

        prompt = format_prompt(sample, include_answer=False)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH - 10,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        generated = tokenizer.decode(
            outputs[0][input_len:],
            skip_special_tokens=True
        )

        predicted = extract_answer(generated)
        predictions[q.id] = predicted   # unique per sample; topic_id would overwrite

    return predictions


# ─────────────────────────────────────────────
# Run on full dev and test sets
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
# Run on dev to check quality
# ─────────────────────────────────────────────

print("Running inference on full dev set ...\n")
dev_predictions = run_inference(data["dev"])

print("\nEvaluating on dev data...")
dev_results = evaluate(dev_predictions, data["dev"])

# ─────────────────────────────────────────────
# Run on test
# ─────────────────────────────────────────────

print("\nRunning inference on test data ...\n")
test_predictions = run_inference(data["test"])
print(f"\nTest predictions generated: {len(test_predictions)}")
print("Sample predictions:", dict(list(test_predictions.items())[:5]))

In [ ]:
# peek at raw question fields
with open("/kaggle/input/datasets/sadegh2002/semeval2026-task12-dataset-main/semeval2026-task12-dataset-main/dev_data/questions.jsonl") as f:
    for i, line in enumerate(f):
        print(json.loads(line).keys())
        if i == 2:
            break

In [ ]:
# ─────────────────────────────────────────────
# Codabench Submission Format
# ─────────────────────────────────────────────
# 1. Run inference on the FULL test set (uncomment and run when ready).
# 2. Build submission CSV: id,prediction (prediction = comma-separated letters, e.g. "A" or "A,C").
# 3. Download the generated file and upload it at:
#    https://www.codabench.org/competitions/12446/#/participate

# Path for the submission file (Kaggle: visible in Output; local: current folder)
SUBMISSION_PATH = "codabench_submission.csv"

def build_submission_csv(predictions: dict[str, list[str]], test_samples: list, out_path: str) -> str:
    """
    Build Codabench-ready CSV from predictions dict.
    predictions: question id -> list of predicted labels, e.g. {"q-2420": ["A", "D"], ...}
    test_samples: list of Sample objects for test_data (order preserved).
    """
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "prediction"])  # header required by most Codabench tasks
        for sample in test_samples:
            qid = sample.question.id
            pred_list = predictions.get(qid, ["A"])  # default "A" if missing
            pred_str = ",".join(sorted(pred_list))   # e.g. "A" or "A,D"
            writer.writerow([qid, pred_str])
    return out_path

# ─────────────────────────────────────────────
# Preview: build from current test_predictions (20 samples) for format verification
# ─────────────────────────────────────────────

# build_submission_csv(test_predictions, data["test"], "codabench_submission_sample.csv")
# print("Sample submission (20 rows) saved: codabench_submission_sample.csv")
# with open("codabench_submission_sample.csv", "r", encoding="utf-8") as f:
#     print("First 10 lines:")
#     for i, line in enumerate(f):
#         if i >= 11:
#             break
#         print(line.rstrip())
# print("\nFor Codabench: run inference on data['test'] (612 samples), then build_submission_csv(..., data['test'], SUBMISSION_PATH).")

In [ ]:
# ─────────────────────────────────────────────
# Run this cell to generate the FINAL Codabench submission file (612 test samples)
# ─────────────────────────────────────────────
# After running: download codabench_submission.csv and upload at
# https://www.codabench.org/competitions/12446/#/participate

print("Running inference on full test set (612 samples)...")
full_test_predictions = run_inference(data["test"])
build_submission_csv(full_test_predictions, data["test"], SUBMISSION_PATH)
print(f"Done. Submission file: {SUBMISSION_PATH}")
print("Download and upload to Codabench.")